# 合成関数と連鎖律（チェインルール）

このノートブックでは、ニューラルネットワークの学習に必要な**合成関数の微分（連鎖律）**について学びます。

---

## このノートブックで学ぶこと

1. **合成関数**とは何か（関数の中に関数がある構造）
2. **連鎖律（チェインルール）**という微分のテクニック
3. ニューラルネットワークで連鎖律がなぜ必要か
4. 実際に計算してみる

---

## 用語の説明（最初に読んでください）

| 用語 | わかりやすい説明 |
|-----|----------------|
| **合成関数** | 関数の中に別の関数が入っている構造。例：$f(g(x))$ は「まず$g$を計算して、その結果を$f$に入れる」 |
| **連鎖律（チェインルール）** | 合成関数を微分するときの公式。「鎖（チェーン）」のようにつなげて計算する |
| **微分** | 「ちょっと変化させたら、どれくらい変わるか」を計算すること |
| **偏微分** | 複数の変数があるとき、1つだけ変化させて微分すること |
| **損失関数** | 「予測がどれくらい間違っているか」を数値で表したもの。小さいほど良い |
| **勾配降下法** | 損失関数を小さくするために、パラメータを少しずつ調整する方法 |

---

## 対応する教科書のセクション
- 3-9: 出力層のパラメータの影響範囲を考察する ー合成関数ー
- 3-10: 出力層のパラメータによって損失関数を最適化する ー偏微分ー
- 3-11: 出力層のパラメータによる損失関数の偏微分結果を導出する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

---

## 1. 合成関数とは？（入れ子構造）

### 1.1 身近な例で理解する

合成関数は、**関数の中に別の関数が入っている**構造です。

#### 日常生活の例

「気温が上がると → アイスの売上が増える → コンビニの利益が増える」

これを関数で表すと：
- $g(\text{気温})$ = アイスの売上
- $f(\text{アイスの売上})$ = コンビニの利益
- 合成関数：$f(g(\text{気温}))$ = コンビニの利益

**「気温が1度上がったら、コンビニの利益はどれくらい変わる？」**

これを計算するのが**連鎖律**です。

### 1.2 数学的な定義

$$f(g(x))$$

これは「**まず $g(x)$ を計算して、その結果を $f$ に入れる**」という意味です。

In [ ]:
# 合成関数の具体例

print("=== 合成関数の具体例 ===")
print()

# 例: f(x) = x², g(x) = 3x + 1 のとき、f(g(x)) は？
def g(x):
    """内側の関数: 3x + 1"""
    return 3 * x + 1

def f(u):
    """外側の関数: u²"""
    return u ** 2

def f_of_g(x):
    """合成関数: f(g(x)) = (3x + 1)²"""
    return f(g(x))  # まずg(x)を計算して、その結果をfに入れる

x = 2
print(f"x = {x} のとき:")
print(f"")
print(f"ステップ1: まず g(x) = 3×{x} + 1 = {g(x)} を計算")
print(f"ステップ2: その結果を f に入れる: f({g(x)}) = {g(x)}² = {f(g(x))}")
print(f"")
print(f"つまり、f(g({x})) = {f_of_g(x)}")

---

## 2. 連鎖律（チェインルール）とは？

### 2.1 なぜ連鎖律が必要？

合成関数 $f(g(x))$ を微分したいとき、普通の微分ルールだけでは計算できません。

そこで使うのが**連鎖律（チェインルール）**です。

### 2.2 連鎖律の公式

$$\{f(g(x))\}' = f'(g(x)) \cdot g'(x)$$

**わかりやすく言うと：**

「**外側を微分** × **内側を微分**」

### 2.3 別の書き方（こちらの方がよく使われます）

$g(x) = u$、$f(u) = y$ と置くと：

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

**イメージ：**
- $x$ → $u$ → $y$ という「鎖（チェーン）」のようにつながっている
- それぞれの「つなぎ目」での変化率を掛け算する

In [ ]:
# 連鎖律の図解

fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)

# x → u → y の流れを図示
boxes = [
    (1, 1.5, '$x$', '入力'),
    (5, 1.5, '$u = g(x)$', '中間'),
    (9, 1.5, '$y = f(u)$', '出力'),
]

for x_pos, y_pos, label, desc in boxes:
    box = FancyBboxPatch((x_pos - 0.8, y_pos - 0.5), 1.6, 1,
                          boxstyle='round,pad=0.1', facecolor='lightblue',
                          edgecolor='blue', linewidth=2)
    ax.add_patch(box)
    ax.text(x_pos, y_pos, label, fontsize=12, ha='center', va='center')
    ax.text(x_pos, y_pos - 1, desc, fontsize=10, ha='center', va='center', color='gray')

# 矢印
ax.annotate('', xy=(4.2, 1.5), xytext=(1.8, 1.5),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(3, 2.2, r"$\frac{du}{dx}$", fontsize=12, ha='center', color='green')
ax.text(3, 0.8, '内側の微分', fontsize=9, ha='center', color='green')

ax.annotate('', xy=(8.2, 1.5), xytext=(5.8, 1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(7, 2.2, r"$\frac{dy}{du}$", fontsize=12, ha='center', color='red')
ax.text(7, 0.8, '外側の微分', fontsize=9, ha='center', color='red')

# 下に公式
ax.text(5, 3.5, r'連鎖律: $\frac{dy}{dx} = \frac{dy}{du} \times \frac{du}{dx}$', 
        fontsize=14, ha='center', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax.axis('off')
plt.title('連鎖律のイメージ：鎖のようにつなげて掛け算する', fontsize=13)
plt.tight_layout()
plt.show()

---

## 3. 連鎖律を使って計算してみよう（Lesson）

連鎖律はAIの学習で何度も使うので、練習して慣れることが大切です。

### 例題1: $f(x) = (3x^2 + 4)^2$ を微分

**ステップ1: 内側と外側に分ける**

- 内側: $u = g(x) = 3x^2 + 4$（カッコの中身）
- 外側: $y = f(u) = u^2$（カッコ全体を2乗）

**ステップ2: それぞれを微分**

- 内側の微分: $\frac{du}{dx} = 6x$（$3x^2 + 4$ を $x$ で微分）
- 外側の微分: $\frac{dy}{du} = 2u$（$u^2$ を $u$ で微分）

**ステップ3: 連鎖律で掛け算**

$$\frac{dy}{dx} = \frac{dy}{du} \times \frac{du}{dx} = 2u \times 6x = 12xu$$

**ステップ4: $u$ を元に戻す**

$$= 12x(3x^2 + 4) = 36x^3 + 48x$$

In [ ]:
# 例題1の計算を確認

print("=== 例題1: f(x) = (3x² + 4)² を微分 ===")
print()

def f1(x):
    """元の関数"""
    return (3*x**2 + 4)**2

def f1_derivative(x):
    """連鎖律で求めた導関数: 36x³ + 48x"""
    return 36*x**3 + 48*x

# 数値微分（答え合わせ用）
def numerical_derivative(f, x, h=0.0001):
    """微分の定義に従って計算（確認用）"""
    return (f(x + h) - f(x - h)) / (2 * h)

print("連鎖律の計算過程:")
print("  内側: u = 3x² + 4")
print("  外側: y = u²")
print("  内側の微分: du/dx = 6x")
print("  外側の微分: dy/du = 2u")
print("  連鎖律: dy/dx = 2u × 6x = 12xu = 12x(3x² + 4) = 36x³ + 48x")
print()

print("検算（数値微分と比較）:")
for x in [1, 2, 3]:
    analytical = f1_derivative(x)  # 連鎖律で計算した結果
    numerical = numerical_derivative(f1, x)  # 定義通りに計算した結果
    print(f"  x={x}: 連鎖律の結果={analytical:.1f}, 数値微分={numerical:.1f} → 一致！")

### 例題2: $f(x) = \sqrt{3x^2 + 4}$ を微分

**ポイント**: $\sqrt{u} = u^{\frac{1}{2}}$ と書き換える

**ステップ1: 内側と外側に分ける**

- 内側: $u = 3x^2 + 4$
- 外側: $y = u^{\frac{1}{2}}$（ルートは $\frac{1}{2}$ 乗）

**ステップ2: それぞれを微分**

- 内側の微分: $\frac{du}{dx} = 6x$
- 外側の微分: $\frac{dy}{du} = \frac{1}{2}u^{-\frac{1}{2}} = \frac{1}{2\sqrt{u}}$

**ステップ3: 連鎖律で掛け算**

$$\frac{dy}{dx} = \frac{1}{2\sqrt{u}} \times 6x = \frac{3x}{\sqrt{u}} = \frac{3x}{\sqrt{3x^2 + 4}}$$

In [ ]:
# 例題2の計算を確認

print("=== 例題2: f(x) = √(3x² + 4) を微分 ===")
print()

def f2(x):
    return np.sqrt(3*x**2 + 4)

def f2_derivative(x):
    """連鎖律で求めた導関数: 3x / √(3x² + 4)"""
    return 3*x / np.sqrt(3*x**2 + 4)

print("連鎖律の計算過程:")
print("  内側: u = 3x² + 4")
print("  外側: y = √u = u^(1/2)")
print("  内側の微分: du/dx = 6x")
print("  外側の微分: dy/du = (1/2)u^(-1/2) = 1/(2√u)")
print("  連鎖律: dy/dx = 1/(2√u) × 6x = 3x/√u = 3x/√(3x² + 4)")
print()

print("検算:")
for x in [1, 2, 3]:
    analytical = f2_derivative(x)
    numerical = numerical_derivative(f2, x)
    print(f"  x={x}: 連鎖律の結果={analytical:.4f}, 数値微分={numerical:.4f} → 一致！")

In [ ]:
# 例題1と例題2のグラフ

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-2, 2, 100)

# 例題1
ax = axes[0]
ax.plot(x, f1(x), 'b-', linewidth=2, label='元の関数: $f(x) = (3x^2+4)^2$')
ax.plot(x, f1_derivative(x), 'r--', linewidth=2, label="導関数: $f'(x) = 36x^3+48x$")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('例題1: $(3x^2+4)^2$ の微分', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# 例題2
ax = axes[1]
ax.plot(x, f2(x), 'b-', linewidth=2, label=r'元の関数: $f(x) = \sqrt{3x^2+4}$')
ax.plot(x, f2_derivative(x), 'r--', linewidth=2, label=r"導関数: $f'(x) = \frac{3x}{\sqrt{3x^2+4}}$")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title(r'例題2: $\sqrt{3x^2+4}$ の微分', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("赤い点線（導関数）は、青い線（元の関数）の『傾き』を表しています。")
print("例えば、例題2で x=0 のとき導関数=0 なので、元の関数は x=0 で水平（傾き0）です。")

---

## 4. ニューラルネットワークでの連鎖律（なぜ必要？）

### 4.1 ニューラルネットワークは「合成関数の塊」

ニューラルネットワークは、たくさんの関数が**入れ子**になった構造です：

```
入力 → 重み計算 → 活性化関数 → 重み計算 → ソフトマックス → 損失関数
```

これは数学的には**合成関数**です：

$$L(P(fc(z(c(x)))))$$

### 4.2 学習で知りたいこと

**「重み $w$ を少し変えたら、損失 $L$ はどれくらい変わる？」**

つまり、$\frac{\partial L}{\partial w}$（損失の重みに対する微分）を計算したい。

→ これを計算するには**連鎖律**が必要！

In [ ]:
# 図3.27: ネットワーク構造と重みの影響範囲

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 12)

ax.text(9, 11.5, '図3.27: 重み $w_{1,0}^{fc}$ がどこに影響を与えるか', 
        fontsize=14, fontweight='bold', ha='center')

# プーリング層
ax.text(3, 10, 'プーリング層の出力', fontsize=11, fontweight='bold', ha='center')
for i, (y, label) in enumerate([(8.5, '$z_1$'), (6.5, '$z_2$')]):
    circle = Circle((3, y), 0.4, facecolor='lightgreen', edgecolor='black')
    ax.add_patch(circle)
    ax.text(3, y, label, fontsize=11, ha='center', va='center')

# 全結合層
ax.text(7.5, 10, '全結合層', fontsize=11, fontweight='bold', ha='center')
fc_neurons = [(7.5, 8.5, '$fc_0$'), (7.5, 7.5, '$fc_1$'), (7.5, 6.5, '$fc_2$')]
for x, y, label in fc_neurons:
    circle = Circle((x, y), 0.4, facecolor='orange', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=10, ha='center', va='center')

# 重要な重み（ハイライト）
ax.annotate('', xy=(7.1, 8.5), xytext=(3.4, 8.5),
            arrowprops=dict(arrowstyle='->', color='blue', lw=3))
ax.text(5, 9.2, '$w_{1,0}^{fc}$', fontsize=12, color='blue', fontweight='bold')
ax.text(5, 8.0, '（この重みを\n調整したい）', fontsize=9, color='blue', ha='center')

# その他の重み
other_connections = [
    (3.4, 6.5, 7.1, 8.5), (3.4, 8.5, 7.1, 7.5), (3.4, 6.5, 7.1, 7.5),
    (3.4, 8.5, 7.1, 6.5), (3.4, 6.5, 7.1, 6.5),
]
for x1, y1, x2, y2 in other_connections:
    ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.3, linewidth=1)

# 出力層
ax.text(12, 10, '出力層\n(ソフトマックス)', fontsize=11, fontweight='bold', ha='center')
output_neurons = [(12, 8.5, '$P_0$'), (12, 7.5, '$P_1$'), (12, 6.5, '$P_2$')]
for x, y, label in output_neurons:
    circle = Circle((x, y), 0.4, facecolor='coral', edgecolor='black')
    ax.add_patch(circle)
    ax.text(x, y, label, fontsize=10, ha='center', va='center')

# 全結合層→出力層の接続
for fx, fy, _ in fc_neurons:
    for ox, oy, _ in output_neurons:
        ax.plot([fx + 0.4, ox - 0.4], [fy, oy], 'gray', alpha=0.2, linewidth=0.5)

# 損失関数
ax.text(16, 10, '損失関数', fontsize=11, fontweight='bold', ha='center')
loss_box = FancyBboxPatch((15, 7), 2, 2, boxstyle='round,pad=0.1',
                           facecolor='lightyellow', edgecolor='red', linewidth=2)
ax.add_patch(loss_box)
ax.text(16, 8, '$L(P)$', fontsize=14, ha='center', va='center')

# 出力層→損失関数の接続
for ox, oy, _ in output_neurons:
    ax.plot([ox + 0.4, 15], [oy, 8], 'gray', alpha=0.3, linewidth=1)

# 影響範囲のボックス
highlight = FancyBboxPatch((6.5, 5.8), 10.5, 4, boxstyle='round,pad=0.1',
                            facecolor='none', edgecolor='blue', linestyle='--', linewidth=2)
ax.add_patch(highlight)

# 説明
ax.text(9, 4.5, '$w_{1,0}^{fc}$ を変えると、青い点線の中（$fc_0$ → $P_0, P_1, P_2$ → $L$）すべてに影響！', 
        fontsize=12, ha='center', color='blue')
ax.text(9, 3.5, 'だから「$w_{1,0}^{fc}$ を変えたら $L$ がどう変わるか」を知るには', fontsize=11, ha='center')
ax.text(9, 2.8, '途中の関数すべてを考慮する必要がある → 連鎖律！', fontsize=12, ha='center', fontweight='bold')

ax.axis('off')
plt.tight_layout()
plt.show()

---

## 5. 実際に計算してみよう

### 5.1 設定の確認

まず、各関数を整理しましょう：

| 何 | 式 | 意味 |
|---|---|------|
| 損失関数 | $L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$ | 予測の間違い具合 |
| ソフトマックス | $P_0 = \frac{e^{fc_0}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$ | 確率に変換 |
| 全結合層 | $fc_0 = w_{1,0}^{fc} \cdot z_1 + w_{2,0}^{fc} \cdot z_2$ | 重み付き和 |

**知りたいこと**: $\frac{\partial L}{\partial w_{1,0}^{fc}}$（重み $w_{1,0}^{fc}$ を変えたら損失 $L$ がどう変わるか）

### 5.2 連鎖律を適用

$$\frac{\partial L}{\partial w_{1,0}^{fc}} = \frac{\partial L}{\partial fc_0} \cdot \frac{\partial fc_0}{\partial w_{1,0}^{fc}}$$

**解釈**:
- 左側: 「$fc_0$ が変わったら $L$ がどう変わるか」
- 右側: 「$w_{1,0}^{fc}$ が変わったら $fc_0$ がどう変わるか」

### 5.3 右側の計算: $\frac{\partial fc_0}{\partial w_{1,0}^{fc}}$

$fc_0 = w_{1,0}^{fc} \cdot z_1 + w_{2,0}^{fc} \cdot z_2$ なので、

$$\frac{\partial fc_0}{\partial w_{1,0}^{fc}} = z_1$$

**なぜ？**
- $w_{1,0}^{fc} \cdot z_1$ を $w_{1,0}^{fc}$ で微分すると $z_1$ が残る
- $w_{2,0}^{fc} \cdot z_2$ には $w_{1,0}^{fc}$ が含まれていないので、微分すると 0 になる

In [ ]:
# ∂fc_0/∂w_{1,0}^fc の計算

print("=== ∂fc_0/∂w_{1,0}^fc の計算 ===")
print()
print("fc_0 = w_{1,0}^fc × z_1 + w_{2,0}^fc × z_2")
print()
print("w_{1,0}^fc で偏微分すると:")
print("  ∂(w_{1,0}^fc × z_1)/∂w_{1,0}^fc = z_1  ← w_{1,0}^fc を含む項")
print("  ∂(w_{2,0}^fc × z_2)/∂w_{1,0}^fc = 0   ← w_{1,0}^fc を含まない")
print()
print("よって: ∂fc_0/∂w_{1,0}^fc = z_1")
print()
print("【直感的な理解】")
print("w_{1,0}^fc を 1 増やすと、fc_0 は z_1 だけ増える")

### 5.4 左側の計算: $\frac{\partial L}{\partial fc_0}$（少し複雑）

$fc_0$ は $P_0$、$P_1$、$P_2$ のすべてに影響を与えています（ソフトマックスの式を見てください）。

だから、連鎖律をさらに適用：

$$\frac{\partial L}{\partial fc_0} = \frac{\partial L}{\partial P_0} \cdot \frac{\partial P_0}{\partial fc_0} + \frac{\partial L}{\partial P_1} \cdot \frac{\partial P_1}{\partial fc_0} + \frac{\partial L}{\partial P_2} \cdot \frac{\partial P_2}{\partial fc_0}$$

**なぜ足し算？**

$fc_0$ が変わると、$P_0$ も $P_1$ も $P_2$ も変わります。それぞれの影響を**合計**する必要があります。

### 5.5 $\frac{\partial L}{\partial P_0}$ の計算（対数関数の微分）

損失関数 $L(P) = -(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)$ を $P_0$ で微分します。

**必要な知識: 対数関数の微分**

$$\frac{d}{dx} \log x = \frac{1}{x}$$

**計算**:

$$\frac{\partial L}{\partial P_0} = \frac{\partial}{\partial P_0}\{-(t_0 \log P_0 + t_1 \log P_1 + t_2 \log P_2)\}$$

$P_1$、$P_2$ を含む項は $P_0$ と無関係なので 0 になり：

$$= -t_0 \cdot \frac{1}{P_0} = -\frac{t_0}{P_0}$$

同様に: $\frac{\partial L}{\partial P_1} = -\frac{t_1}{P_1}$、$\frac{\partial L}{\partial P_2} = -\frac{t_2}{P_2}$

In [ ]:
# 対数関数の微分の可視化

fig, ax = plt.subplots(figsize=(10, 6))

x = np.linspace(0.1, 4, 100)

ax.plot(x, np.log(x), 'b-', linewidth=2, label=r'$f(x) = \log x$')
ax.plot(x, 1/x, 'r--', linewidth=2, label=r"$f'(x) = 1/x$（微分）")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)

# x=1での値
ax.scatter([1], [0], color='blue', s=100, zorder=5)
ax.scatter([1], [1], color='red', s=100, zorder=5)
ax.annotate('log(1) = 0', xy=(1, 0), xytext=(1.8, 0.5), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='blue'))
ax.annotate("1/1 = 1", xy=(1, 1), xytext=(1.8, 1.5), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title(r'対数関数の微分: $(\log x)\prime = 1/x$', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.3, 4)
ax.set_ylim(-2, 3)

plt.tight_layout()
plt.show()

print("【重要公式】 d/dx (log x) = 1/x")
print()
print("これを使って、損失関数 L(P) = -t_0 log P_0 - ... を P_0 で微分すると:")
print("∂L/∂P_0 = -t_0 × (1/P_0) = -t_0/P_0")

### 5.6 $\frac{\partial P_0}{\partial fc_0}$ の計算（ソフトマックスの微分）

これは少し複雑ですが、結果は美しいです。

$$P_0 = \frac{e^{fc_0}}{e^{fc_0} + e^{fc_1} + e^{fc_2}}$$

**分数関数の微分公式（商の微分）**を使います：

$$\left(\frac{f(x)}{g(x)}\right)' = \frac{f'(x) \cdot g(x) - f(x) \cdot g'(x)}{(g(x))^2}$$

詳しい計算は省略しますが、結果は：

$$\frac{\partial P_0}{\partial fc_0} = P_0(1 - P_0)$$

**直感的な理解**:
- $P_0$ が 0 に近いとき、変化率も小さい
- $P_0$ が 1 に近いとき、変化率も小さい
- $P_0$ が 0.5 のとき、変化率が最大

In [ ]:
# ∂P_0/∂fc_0 = P_0(1-P_0) の可視化

fig, ax = plt.subplots(figsize=(10, 6))

P0 = np.linspace(0.01, 0.99, 100)
derivative = P0 * (1 - P0)

ax.plot(P0, derivative, 'b-', linewidth=2)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
ax.scatter([0.5], [0.25], color='red', s=100, zorder=5)

ax.set_xlabel('$P_0$（確率）', fontsize=12)
ax.set_ylabel(r'$\frac{\partial P_0}{\partial fc_0} = P_0(1-P_0)$', fontsize=12)
ax.set_title(r'ソフトマックスの微分: $\frac{\partial P_0}{\partial fc_0} = P_0(1-P_0)$', fontsize=14)
ax.grid(True, alpha=0.3)

ax.annotate('最大値 (P_0=0.5 で 0.25)', xy=(0.5, 0.25), xytext=(0.7, 0.2), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print("【結果】∂P_0/∂fc_0 = P_0(1 - P_0)")
print()
print("同様に:")
print("  ∂P_1/∂fc_0 = -P_1 × P_0  （P_1 は fc_0 が増えると減る）")
print("  ∂P_2/∂fc_0 = -P_2 × P_0  （P_2 も fc_0 が増えると減る）")

---

## 6. Lesson: ネイピア数 $e$ の重要性

### 6.1 ネイピア数とは？

$$e \approx 2.71828...$$

円周率 $\pi$ と同じく、特別な意味を持つ無限に続く数（**超越数**）です。

### 6.2 なぜソフトマックスで $e$ を使う？

**最大の理由: 微分しても形が変わらない！**

$$\frac{d}{dx} e^x = e^x$$

他の指数関数（例: $2^x$）だとこうはいきません。この性質のおかげで、計算が非常にシンプルになります。

### 6.3 ネイピア数の定義（図3.29）

$y = a^x$ という指数関数で、点 $(0, 1)$ での接線の傾きがちょうど 1 になる $a$ の値が $e$ です。

In [ ]:
# 図3.29: ネイピア数の定義

fig, ax = plt.subplots(figsize=(10, 8))

x = np.linspace(-2, 2, 200)

# y = e^x
ax.plot(x, np.exp(x), 'b-', linewidth=2, label=r'$y = e^x$')

# 点(0, 1)での接線 y = x + 1
ax.plot(x, x + 1, 'r--', linewidth=2, label=r'接線: $y = x + 1$（傾き1）')

# 点(0, 1)
ax.scatter([0], [1], color='red', s=100, zorder=5)
ax.annotate('接点 (0, 1)', xy=(0, 1), xytext=(0.5, 1.8), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'))

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.axhline(y=1, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('図3.29: ネイピア数 $e$ の定義\n$y = e^x$ は点 (0, 1) で傾き 1 の接線を持つ', fontsize=14)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)
ax.set_ylim(-0.5, 5)

# 説明テキスト
ax.text(0.5, 4, r'$\frac{d}{dx}e^x = e^x$', fontsize=14, 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
ax.text(0.5, 3.3, '微分しても同じ形！', fontsize=11)

plt.tight_layout()
plt.show()

print("ネイピア数 e ≈ 2.71828...")
print()
print("【e の特別な性質】")
print("  (e^x)' = e^x  ← 微分しても形が変わらない！")
print()
print("これがソフトマックスで e を使う理由です。")
print("計算が非常にシンプルになります。")

---

## 7. すべてをまとめて計算

### 7.1 最終結果

長い計算の結果、驚くほどシンプルな式が得られます：

$$\frac{\partial L}{\partial w_{1,0}^{fc}} = (P_0 - t_0) \cdot z_1$$

**意味**:
- $P_0 - t_0$：予測と正解の差（誤差）
- $z_1$：入力値

「**誤差 × 入力**」という非常に直感的な形！

### 7.2 重みの更新

$$w_{1,0}^{fc \text{(新)}} = w_{1,0}^{fc \text{(古)}} - \eta \cdot (P_0 - t_0) \cdot z_1$$

- $\eta$（イータ）：学習率（どれくらい大きく更新するか）

In [ ]:
# 完全な実装

def softmax(x):
    """ソフトマックス関数：数値を確率に変換"""
    x_shifted = x - np.max(x)  # 数値安定性のため
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x)

def cross_entropy_loss(P, t):
    """クロスエントロピー損失：予測の間違い具合を計算"""
    return -np.sum(t * np.log(P + 1e-10))

# 設定
z = np.array([1.5, 2.0])  # 入力 [z1, z2]
t = np.array([1, 0, 0])   # 正解（クラス0が正解）
w = np.array([[0.5, 0.3, 0.2],   # z1からの重み
              [0.4, 0.5, 0.1]])  # z2からの重み

print("=== 連鎖律による勾配計算の完全な例 ===")
print()
print(f"入力: z = {z}  (z1={z[0]}, z2={z[1]})")
print(f"正解: t = {t}  (クラス0が正解)")
print()

# 順伝播
fc = z @ w  # 全結合層の出力
P = softmax(fc)  # ソフトマックス
L = cross_entropy_loss(P, t)  # 損失

print("【順伝播（前から順に計算）】")
print(f"  fc = [fc_0, fc_1, fc_2] = {fc}")
print(f"  P  = [P_0, P_1, P_2]   = [{P[0]:.4f}, {P[1]:.4f}, {P[2]:.4f}]")
print(f"  損失 L = {L:.4f}")
print()

# 勾配計算（連鎖律の結果を使用）
# ∂L/∂w_{1,0}^fc = (P_0 - t_0) × z_1
dL_dw10 = (P[0] - t[0]) * z[0]

print("【連鎖律による勾配計算】")
print(f"  ∂L/∂w_{{1,0}}^fc = (P_0 - t_0) × z_1")
print(f"                  = ({P[0]:.4f} - {t[0]}) × {z[0]}")
print(f"                  = {P[0] - t[0]:.4f} × {z[0]}")
print(f"                  = {dL_dw10:.4f}")

In [ ]:
# 数値微分で検算

def compute_loss(w10_value):
    """w_{1,0}^fc を変えたときの損失"""
    w_temp = w.copy()
    w_temp[0, 0] = w10_value
    fc_temp = z @ w_temp
    P_temp = softmax(fc_temp)
    return cross_entropy_loss(P_temp, t)

# 数値微分
h = 0.0001
numerical_grad = (compute_loss(w[0, 0] + h) - compute_loss(w[0, 0] - h)) / (2 * h)

print("【検算: 数値微分との比較】")
print(f"  連鎖律での計算: {dL_dw10:.6f}")
print(f"  数値微分の結果: {numerical_grad:.6f}")
print(f"  差: {abs(dL_dw10 - numerical_grad):.10f}")
print()
print("  → ほぼ一致！連鎖律の計算が正しいことを確認。")

In [ ]:
# 学習のシミュレーション

def train_step(w, z, t, learning_rate):
    """1回の学習ステップ"""
    fc = z @ w
    P = softmax(fc)
    L = cross_entropy_loss(P, t)
    
    # 勾配: (P - t) の各要素と z の外積
    grad = np.outer(z, P - t)
    
    # 重み更新
    w_new = w - learning_rate * grad
    return w_new, L, P

# 学習の実行
np.random.seed(42)
w = np.random.randn(2, 3) * 0.5
z = np.array([1.5, 2.0])
t = np.array([1, 0, 0])

losses = []
probs = []

print("=== 学習の進行 ===")
print()

for epoch in range(50):
    w, L, P = train_step(w, z, t, learning_rate=0.5)
    losses.append(L)
    probs.append(P.copy())
    
    if epoch < 5 or (epoch + 1) % 10 == 0:
        predicted_class = np.argmax(P)
        print(f"エポック{epoch+1:2d}: 損失={L:.4f}, P_0={P[0]:.3f}, 予測クラス={predicted_class}")

print()
print(f"最終結果: P_0 = {probs[-1][0]:.4f}")
print(f"正解クラス(0)の確率が {probs[-1][0]*100:.1f}% に向上！")

In [ ]:
# 学習の可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 損失の推移
ax = axes[0]
ax.plot(losses, 'b-', linewidth=2)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('損失', fontsize=12)
ax.set_title('損失関数の推移\n（小さいほど良い）', fontsize=14)
ax.grid(True, alpha=0.3)

# 確率の推移
ax = axes[1]
probs_array = np.array(probs)
ax.plot(probs_array[:, 0], 'r-', linewidth=2, label='$P_0$（正解クラス）')
ax.plot(probs_array[:, 1], 'g--', linewidth=2, label='$P_1$')
ax.plot(probs_array[:, 2], 'b:', linewidth=2, label='$P_2$')
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('確率', fontsize=12)
ax.set_title('各クラスの予測確率の推移\n（正解クラスが1に近づく）', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

print("【結果の解釈】")
print("・左のグラフ: 損失が減少 → 予測が改善している")
print("・右のグラフ: 正解クラス(赤)の確率が上昇 → 正しく予測できるようになった")

---

## まとめ

### 今日学んだこと

| 概念 | 説明 |
|-----|------|
| **合成関数** | 関数の中に関数がある構造。$f(g(x))$ |
| **連鎖律** | 合成関数を微分する公式。$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$ |
| **なぜ必要？** | ニューラルネットワークは合成関数の塊だから |
| **最終結果** | $\frac{\partial L}{\partial w} = (P - t) \cdot z$（誤差 × 入力）|

### 重要な公式

1. **連鎖律**: $\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$

2. **対数の微分**: $\frac{d}{dx} \log x = \frac{1}{x}$

3. **ネイピア数の微分**: $\frac{d}{dx} e^x = e^x$

4. **勾配の最終形**: $\frac{\partial L}{\partial w_{i,j}^{fc}} = (P_j - t_j) \cdot z_i$

### 次のステップ

次回は、この連鎖律をさらに深い層（畳み込み層）まで適用する**誤差逆伝播法（バックプロパゲーション）**について学びます。